# Mega-Sena MVP

Notebook de pratica: reproduz a mesma base sintetica usada no projeto original (repositorio `andrelmsunb/PredicaoMegasena`), refaz as principais analises exploratorias e treina um unico modelo preditivo (Random Forest) por cima dessas analises.

Repositorio de referencia: https://github.com/andrelmsunb/PredicaoMegasena

Aluno: Matheus Victor Costa Cabral - 221021035

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_style('whitegrid')
%matplotlib inline

## 1. Geracao da base de dados (sintetica, seed = 42)

Mesma logica do notebook original: numeros de 1 a 60, com um conjunto de numeros levemente mais provaveis (peso 1.1), gerando 2900 concursos simulados com estatisticas derivadas (soma, pares/impares, consecutivos, distribuicao por dezena e por terminacao).

In [ ]:
class MegaSenaDataCollector:
    '''
    Gera uma base sintetica de sorteios da Mega-Sena, com a mesma
    semente e regras do repositorio original, para fins de pratica.
    '''

    def __init__(self):
        self.data = None

    def create_sample_data(self, n_records=2900):
        print('Gerando', n_records, 'registros simulados...')
        np.random.seed(42)

        data = []
        hot_numbers = [5, 9, 13, 15, 20, 23, 25, 27, 29, 30, 31, 34, 41, 46, 47, 49, 52, 53, 55]

        for i in range(1, n_records + 1):
            weights = np.ones(60)
            for num in hot_numbers:
                weights[num - 1] *= 1.1
            weights = weights / weights.sum()

            numbers = np.random.choice(range(1, 61), size=6, replace=False, p=weights)
            numbers = sorted(numbers)

            soma = sum(numbers)
            pares = sum(1 for x in numbers if x % 2 == 0)
            impares = 6 - pares
            consecutivos = self._count_consecutives(numbers)
            diff_max_min = max(numbers) - min(numbers)

            dezenas = [0] * 6
            for num in numbers:
                idx = (num - 1) // 10
                if idx < 6:
                    dezenas[idx] += 1

            terminacoes = [0] * 10
            for num in numbers:
                terminacoes[num % 10] += 1

            record = {
                'concurso': i,
                'data': str(2023) + '-' + str((i % 12) + 1).zfill(2) + '-' + str((i % 28) + 1).zfill(2),
                'numero1': numbers[0], 'numero2': numbers[1], 'numero3': numbers[2],
                'numero4': numbers[3], 'numero5': numbers[4], 'numero6': numbers[5],
                'soma_numeros': soma,
                'numeros_pares': pares,
                'numeros_impares': impares,
                'numeros_consecutivos': consecutivos,
                'diff_max_min': diff_max_min,
                'ganhadores': np.random.randint(0, 5),
                'premio': np.random.randint(1000000, 50000000),
                'apostas': np.random.randint(10000000, 50000000),
            }

            faixas = [(1, 10), (11, 20), (21, 30), (31, 40), (41, 50), (51, 60)]
            for j, (start, end) in enumerate(faixas):
                record['dezena_' + str(start) + '_' + str(end)] = dezenas[j]
            for j in range(10):
                record['termina_' + str(j)] = terminacoes[j]

            data.append(record)

        self.data = pd.DataFrame(data)
        print(len(self.data), 'registros gerados com sucesso.')
        return self.data

    @staticmethod
    def _count_consecutives(numbers):
        s = sorted(numbers)
        return sum(1 for i in range(len(s) - 1) if s[i + 1] - s[i] == 1)


collector = MegaSenaDataCollector()
df = collector.create_sample_data(n_records=2900)
df.head()

In [ ]:
print('Formato da base:', df.shape)
df.info()

## 2. Analise exploratoria

Mesmas dimensoes analisadas no repositorio original: frequencia dos numeros, distribuicao da soma, paridade, numeros consecutivos, distribuicao por dezena, por terminacao e correlacao entre as variaveis derivadas.

In [ ]:
numero_cols = ['numero1', 'numero2', 'numero3', 'numero4', 'numero5', 'numero6']
frequencias = pd.Series(df[numero_cols].values.ravel()).value_counts().sort_index()

plt.figure(figsize=(14, 5))
plt.bar(frequencias.index, frequencias.values, color='steelblue')
plt.title('Frequencia de cada numero sorteado (1 a 60)')
plt.xlabel('Numero')
plt.ylabel('Vezes sorteado')
plt.show()

print('Top 10 numeros mais frequentes:')
print(frequencias.sort_values(ascending=False).head(10))

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(df['soma_numeros'], kde=True, bins=30, color='darkorange')
plt.title('Distribuicao da soma dos 6 numeros sorteados')
plt.xlabel('Soma')
plt.show()

media_soma = round(df['soma_numeros'].mean(), 2)
std_soma = round(df['soma_numeros'].std(), 2)
print('Media:', media_soma, '| Desvio padrao:', std_soma)

amostra = df['soma_numeros'].sample(min(len(df), 5000), random_state=42)
stat, p = stats.shapiro(amostra)
print('Teste de normalidade (Shapiro-Wilk): estatistica =', round(stat, 4), '| p-valor =', round(p, 4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df['numeros_pares'].value_counts().sort_index().plot(kind='bar', ax=axes[0], color='seagreen')
axes[0].set_title('Quantidade de numeros pares por concurso')
axes[0].set_xlabel('Numeros pares (de 6)')

df['numeros_consecutivos'].value_counts().sort_index().plot(kind='bar', ax=axes[1], color='indianred')
axes[1].set_title('Quantidade de numeros consecutivos por concurso')
axes[1].set_xlabel('Pares consecutivos')

plt.tight_layout()
plt.show()

In [ ]:
dezena_cols = ['dezena_1_10', 'dezena_11_20', 'dezena_21_30', 'dezena_31_40', 'dezena_41_50', 'dezena_51_60']
termina_cols = ['termina_' + str(j) for j in range(10)]

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

df[dezena_cols].mean().plot(kind='bar', ax=axes[0], color='slateblue')
axes[0].set_title('Media de numeros sorteados por faixa de dezena')
axes[0].set_ylabel('Media por concurso')

df[termina_cols].mean().plot(kind='bar', ax=axes[1], color='goldenrod')
axes[1].set_title('Media de numeros sorteados por terminacao (0 a 9)')
axes[1].set_ylabel('Media por concurso')

plt.tight_layout()
plt.show()

In [ ]:
colunas_corr = ['soma_numeros', 'numeros_pares', 'numeros_consecutivos', 'diff_max_min', 'ganhadores', 'premio', 'apostas']

plt.figure(figsize=(9, 7))
sns.heatmap(df[colunas_corr].corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlacao entre variaveis derivadas do sorteio')
plt.tight_layout()
plt.show()

## 3. Modelo preditivo (Random Forest)

O alvo escolhido e `soma_numeros`, a partir das demais estatisticas derivadas do mesmo concurso (paridade, consecutivos, dispersao, distribuicao por dezena e por terminacao). E o mesmo tipo de exercicio do repositorio original: um Random Forest Regressor treinado sobre as caracteristicas de cada sorteio, util para praticar o pipeline de ML (split, escala, treino, avaliacao), e nao uma predicao real de sorteios futuros.

In [ ]:
feature_cols = ['numeros_pares', 'numeros_consecutivos', 'diff_max_min'] + dezena_cols + termina_cols

X = df[feature_cols]
y = df['soma_numeros']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

modelo = RandomForestRegressor(n_estimators=200, random_state=42)
modelo.fit(X_train_scaled, y_train)

y_pred = modelo.predict(X_test_scaled)

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print('MAE:', round(mae, 3))
print('MSE:', round(mse, 3))
print('R2:', round(r2, 3))

In [ ]:
importancias = pd.Series(modelo.feature_importances_, index=feature_cols).sort_values(ascending=False)

plt.figure(figsize=(10, 6))
importancias.head(15).plot(kind='barh', color='teal')
plt.title('Importancia das variaveis no Random Forest')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 4. Sugestao final de numeros (estatistica, nao e uma predicao)

Sorteia 6 numeros ponderados pela frequencia observada na base sintetica gerada acima. Isso e apenas uma amostragem estatistica sobre dados simulados, no mesmo espirito do notebook original: um exercicio de pratica, sem qualquer garantia de resultado em um sorteio real.

In [ ]:
def sugerir_numeros(frequencias, quantidade=6, seed=None):
    pesos = frequencias / frequencias.sum()
    rng = np.random.RandomState(seed)
    sugestao = rng.choice(frequencias.index, size=quantidade, replace=False, p=pesos.values)
    return sorted(sugestao.tolist())


sugestao = sugerir_numeros(frequencias, seed=7)
print('Sugestao de numeros (base em frequencia historica simulada):', sugestao)
print()
print('Lembrete: a Mega-Sena e um sorteio aleatorio. Nenhuma analise estatistica ou modelo de ML muda a probabilidade de acerto em um sorteio real.')